In [ ]:
# ===== NB17 | CELL 1: install LIME + GPU check =====
from google.colab import drive
drive.mount('/content/drive')
!pip install -q lime
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - change the runtime to T4 GPU')
print('CELL 1 DONE')

In [ ]:
# ===== NB17 | CELL 2: LIME explanations for 20 comments (the brief asks for 15-20) =====
# WHAT: choose 20 short comments from the internal test set (correct flags, false alarms, misses, clean comments,
#       and two false alarms that mention an identity word), and explain each with LIME using the FROZEN system.
# WHY:  a moderation tool must be able to say WHY it flagged something. The mix lets us show both
#       what works and what fails.
import sys, importlib, json, os, re
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
sys.path.insert(0, f'{PROJECT_DIR}/src')
import toxlib, nnlib, runlib
for m in (toxlib, nnlib, runlib): importlib.reload(m)
from toxlib import LABELS, load_split
import numpy as np, pandas as pd
from lime.lime_text import LimeTextExplainer
for d in ['explainability', 'tables', 'graphs']:
    os.makedirs(f'{PROJECT_DIR}/results/{d}', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results/explainability/lime_html', exist_ok=True)

cfg = json.load(open(f'{PROJECT_DIR}/results/final/final_config.json')); thr = cfg['thresholds']
df = load_split(PROJECT_DIR, 'test_internal')
Pf = np.load(f'{PROJECT_DIR}/results/final/final_test_internal.npy')
for i, l in enumerate(LABELS):
    df['p_' + l] = Pf[:, i]
df['words'] = df.comment_text.fillna('').str.split().str.len()
short = df[(df.words >= 4) & (df.words <= 60)]               # short comments give readable explanations
IDENT = r'\b(?:muslim|hindu|christian|jewish|sikh|buddhist|atheist|gay|lesbian|bisexual|homosexual|transgender|black|white|asian|latino|indian|american|dalit|brahmin|blind|deaf|disabled)\b'

chosen, used = [], set()
def pick(label, kind, n):
    y, p, t = short[label] == 1, short['p_' + label], thr[label]
    mask = {'TP': y & (p >= t), 'FP': ~y & (p >= t), 'FN': y & (p < t),
            'TN': (short[LABELS].sum(axis=1) == 0) & (p < 0.5 * t),
            'FP_IDENT': ~y & (p >= t) & short.comment_text.fillna('').str.contains(IDENT, case=False, regex=True)}[kind]
    pool = short[mask & ~short.id.isin(used)]
    s = pool.sample(min(n, len(pool)), random_state=42).copy()
    s['label'], s['kind'], s['prob'] = label, kind, s['p_' + label].round(3)
    used.update(s.id); chosen.append(s)
    if len(s) < n: print(f'  note: only {len(s)} of {n} found for {label} {kind}')
PLAN = [('toxic', 'TP', 1), ('obscene', 'TP', 1), ('insult', 'TP', 1), ('threat', 'TP', 1), ('identity_hate', 'TP', 1), ('severe_toxic', 'TP', 1),
        ('toxic', 'FP', 3), ('threat', 'FP', 1), ('identity_hate', 'FP', 1),
        ('toxic', 'FN', 2), ('threat', 'FN', 1), ('identity_hate', 'FN', 1),
        ('toxic', 'TN', 3), ('toxic', 'FP_IDENT', 2)]
for lab, kind, n in PLAN: pick(lab, kind, n)
C = pd.concat(chosen).reset_index(drop=True)
print(f'{len(C)} comments chosen for explanation')

ens = runlib.Ensemble(PROJECT_DIR, cfg['runs'])
explainer = LimeTextExplainer(class_names=LABELS, random_state=42)
records = []
for k, r in C.iterrows():
    idx = LABELS.index(r.label)
    exp = explainer.explain_instance(r.comment_text, ens.predict, labels=(idx,), num_features=8, num_samples=1000)
    words = exp.as_list(label=idx)
    exp.save_to_file(f'{PROJECT_DIR}/results/explainability/lime_html/{k:02d}_{r.label}_{r.kind}.html', labels=[idx])
    records.append({'n': k, 'id': r.id, 'label': r.label, 'kind': r.kind, 'prob': r.prob, 'threshold': thr[r.label],
                    'true_label': int(r[r.label]), 'text': r.comment_text.replace('\n', ' '),
                    'top_words': '; '.join(f'{w} ({s:+.2f})' for w, s in words[:6]), 'words_json': json.dumps(words)})
    print(f"\n[{k}] {r.label} | {r.kind} | prob={r.prob:.2f} (threshold {thr[r.label]}) | true={int(r[r.label])}")
    print('  text:', r.comment_text[:140].replace('\n', ' '))
    print('  top words:', '; '.join(f'{w} ({s:+.2f})' for w, s in words[:6]))
T14 = pd.DataFrame(records)
T14.to_csv(f'{PROJECT_DIR}/results/tables/T14_lime_explanations.csv', index=False)
print('\nCELL 2 DONE')

In [ ]:
# ===== NB17 | CELL 3: faithfulness check + explanation plots =====
# WHAT: for every comment the model flagged, delete LIME's top-5 words and measure the drop in probability,
#       and compare with deleting 5 random words (20 random draws). Also plots F19-F21.
# WHY:  an explanation is only useful if it reflects what the model really uses. This test checks it.
import sys, importlib, json, os, re
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
sys.path.insert(0, f'{PROJECT_DIR}/src')
import toxlib, nnlib, runlib
for m in (toxlib, nnlib, runlib): importlib.reload(m)
from toxlib import LABELS
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt
for d in ['explainability', 'tables', 'graphs']:
    os.makedirs(f'{PROJECT_DIR}/results/{d}', exist_ok=True)

cfg = json.load(open(f'{PROJECT_DIR}/results/final/final_config.json')); thr = cfg['thresholds']
T14 = pd.read_csv(f'{PROJECT_DIR}/results/tables/T14_lime_explanations.csv')
ens = runlib.Ensemble(PROJECT_DIR, cfg['runs'])
rng = np.random.RandomState(42)

def delete_words(text, words):
    for w in words:
        text = re.sub(rf'\b{re.escape(w)}\b', ' ', text, flags=re.I)
    return re.sub(r'\s+', ' ', text).strip()

# faithfulness: only for comments the model flagged (prob >= threshold)
rows = []
for _, r in T14.iterrows():
    if r.prob < r.threshold:
        continue
    idx = LABELS.index(r.label); words = json.loads(r.words_json)
    top = [w for w, s in sorted(words, key=lambda x: -x[1]) if s > 0][:5]
    if not top:
        continue
    uniq = sorted(set(re.findall(r'\w+', r.text)))
    rand_sets = [list(rng.choice(uniq, size=min(len(top), len(uniq)), replace=False)) for _ in range(20)]
    texts = [r.text, delete_words(r.text, top)] + [delete_words(r.text, rs) for rs in rand_sets]
    p = ens.predict(texts)[:, idx]
    rows.append({'n': r.n, 'label': r.label, 'kind': r.kind, 'p_original': round(p[0], 3), 'p_after_top5_removed': round(p[1], 3),
                 'drop_top5': round(p[0] - p[1], 3), 'drop_random5_mean': round(p[0] - p[2:].mean(), 3),
                 'flipped_by_top5': bool(p[0] >= r.threshold > p[1]), 'flipped_by_random_share': round(float(np.mean(p[2:] < r.threshold)), 2)})
F = pd.DataFrame(rows)
F.to_csv(f'{PROJECT_DIR}/results/tables/T15_faithfulness.csv', index=False)
print('T15 faithfulness (comments the model flagged):'); print(F.to_string(index=False))
summary = {'n_examples': int(len(F)), 'mean_drop_top5': round(float(F.drop_top5.mean()), 3),
           'mean_drop_random5': round(float(F.drop_random5_mean.mean()), 3),
           'share_top5_beats_random': round(float((F.drop_top5 > F.drop_random5_mean).mean()), 2),
           'flips_by_top5': int(F.flipped_by_top5.sum()), 'expected_flips_by_random': round(float(F.flipped_by_random_share.sum()), 1)}
json.dump(summary, open(f'{PROJECT_DIR}/results/final/faithfulness_summary.json', 'w'), indent=2)
print('\nSummary:', summary)

# F19: LIME bar plots for the first 6 flagged examples
S = T14[T14.prob >= T14.threshold].head(6)
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for a, (_, r) in zip(axes.ravel(), S.iterrows()):
    w = sorted(json.loads(r.words_json), key=lambda x: x[1])
    a.barh([x[0] for x in w], [x[1] for x in w], color=['#D9776B' if x[1] > 0 else '#4C72B0' for x in w])
    a.set_title(f"{r.label} | {r.kind} | p={r.prob:.2f}\n{r.text[:55]}...", fontsize=9); a.axvline(0, color='k', lw=.5)
plt.suptitle('F19: LIME explanations (red pushes toward the label, blue pushes away)'); plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/results/graphs/F19_lime_examples.png', dpi=150); plt.show()
# F20: deleting top-5 words vs random words
plt.figure(figsize=(5, 4))
plt.bar(['top-5 LIME words', '5 random words'], [F.drop_top5.mean(), F.drop_random5_mean.mean()], color=['#D9776B', '#9AA5B1'])
plt.scatter(np.zeros(len(F)), F.drop_top5, color='k', s=12); plt.scatter(np.ones(len(F)), F.drop_random5_mean, color='k', s=12)
plt.ylabel('drop in predicted probability'); plt.title('F20: faithfulness check'); plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/results/graphs/F20_faithfulness.png', dpi=150); plt.show()

# F21: global view = top Logistic Regression words per label
ck = f'{PROJECT_DIR}/models/saved_checkpoints'
word = joblib.load(f'{ck}/word_tfidf_vectorizer.pkl'); clf = joblib.load(f'{ck}/A1_logistic_regression.pkl')
names = np.array(word.get_feature_names_out())
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
rows = []
for i, (l, a) in enumerate(zip(LABELS, axes.ravel())):
    w = clf.estimators_[i].coef_[0][:len(names)]; top = np.argsort(-w)[:10]
    a.barh(names[top][::-1], w[top][::-1], color='#4C72B0'); a.set_title(l)
    rows += [{'label': l, 'word': names[j], 'coef': round(float(w[j]), 3)} for j in top]
plt.suptitle('F21: words the Logistic Regression baseline links most to each label'); plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/results/graphs/F21_lr_coefficients.png', dpi=150); plt.show()
pd.DataFrame(rows).to_csv(f'{PROJECT_DIR}/results/tables/T16_lr_top_words.csv', index=False)
print('CELL 3 DONE')